# 数据统计

In [2]:
import pandas as pd
import json

from collections import defaultdict

In [3]:
import sys
sys.path.append('..')

## 原始数据

In [4]:
df_raw = pd.read_excel('data/mayday_songs.xlsx', sheet_name='Sheet1')
df_raw['song_name'] = df_raw['song_name'].astype(str)
df_raw

,album_order,album_id,album_name,album_type,release_date,song_order,song_id,song_name,album_fixed,has_lyric,is_duplicate
0,1,38315,第一张创作专辑,录音室专辑,1999-07-07,1,386925,疯狂世界,第一张创作专辑,1,0
1,1,38315,第一张创作专辑,录音室专辑,1999-07-07,2,386927,拥抱,第一张创作专辑,1,0
2,1,38315,第一张创作专辑,录音室专辑,1999-07-07,3,386929,透露,第一张创作专辑,1,0
3,1,38315,第一张创作专辑,录音室专辑,1999-07-07,4,386930,生活,第一张创作专辑,1,0
4,1,38315,第一张创作专辑,录音室专辑,1999-07-07,5,386931,爱情的模样,第一张创作专辑,1,0
...,...,...,...,...,...,...,...,...,...,...,...
185,11,2740205,步步 自选作品辑,精选辑,2013-12-30,26,28181124,雌雄同体,步步 自选作品辑,1,1
186,11,2740205,步步 自选作品辑,精选辑,2013-12-30,27,28181126,生命有一种绝对,步步 自选作品辑,1,1
187,11,2740205,步步 自选作品辑,精选辑,2013-12-30,28,28181128,诺亚方舟,步步 自选作品辑,1,1
188,11,2740205,步步 自选作品辑,精选辑,2013-12-30,29,28181130,我心中尚未崩坏的地方,步步 自选作品辑,1,1


In [5]:
df_lyric_raw = pd.read_csv('output/mayday_lyric_word.csv')
df_lyric_raw

,Unnamed: 0,song_id,词,词性,频数,album_order,album_id,album_name,album_type,release_date,song_order,song_name,album_fixed,has_lyric,is_duplicate
0,0,386925,我,r,18,1,38315,第一张创作专辑,录音室专辑,1999-07-07,1,疯狂世界,第一张创作专辑,1,0
1,1,386925,好想,v,18,1,38315,第一张创作专辑,录音室专辑,1999-07-07,1,疯狂世界,第一张创作专辑,1,0
2,2,386925,那么,r,12,1,38315,第一张创作专辑,录音室专辑,1999-07-07,1,疯狂世界,第一张创作专辑,1,0
3,3,386925,多,m,12,1,38315,第一张创作专辑,录音室专辑,1999-07-07,1,疯狂世界,第一张创作专辑,1,0
4,4,386925,的,uj,12,1,38315,第一张创作专辑,录音室专辑,1999-07-07,1,疯狂世界,第一张创作专辑,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13380,13380,28181110,长大,ns,1,11,2740205,步步 自选作品辑,精选辑,2013-12-30,19,盛夏光年,步步 自选作品辑,1,0
13381,13381,28181110,难道,d,1,11,2740205,步步 自选作品辑,精选辑,2013-12-30,19,盛夏光年,步步 自选作品辑,1,0
13382,13382,28181110,人,n,1,11,2740205,步步 自选作品辑,精选辑,2013-12-30,19,盛夏光年,步步 自选作品辑,1,0
13383,13383,28181110,必经,d,1,11,2740205,步步 自选作品辑,精选辑,2013-12-30,19,盛夏光年,步步 自选作品辑,1,0


## 专辑数据

In [6]:
df_album = df_raw[[
    'album_id', 'album_name', 'album_type', 'release_date', 'album_fixed'
]].copy()
df_album['release_date'] = df_album['release_date'].astype(str)
df_album = df_album.drop_duplicates().reset_index()
df_album

,index,album_id,album_name,album_type,release_date,album_fixed
0,0,38315,第一张创作专辑,录音室专辑,1999-07-07,第一张创作专辑
1,12,38308,爱情万岁,录音室专辑,2000-07-07,爱情万岁
2,24,38297,人生海海,录音室专辑,2001-07-06,人生海海
3,36,38276,时光机,录音室专辑,2003-11-11,时光机
4,51,38259,神的孩子都在跳舞,录音室专辑,2004-11-05,神的孩子都在跳舞
5,64,38241,为爱而生,录音室专辑,2006-12-29,为爱而生
6,77,38235,后青春期的诗,录音室专辑,2008-10-23,后青春期的诗
7,89,2040001,第二人生 (明日版),录音室专辑,2011-12-16,第二人生
8,103,38214,第二人生 (末日版),录音室专辑,2011-12-16,第二人生
9,117,34746073,自传,录音室专辑,2016-07-21,自传


In [7]:
# 歌曲计数
songs_num = df_raw['album_name'].value_counts()
songs_num

album_name
知足 最真杰作选      30
步步 自选作品辑      30
时光机           15
第二人生 (明日版)    14
第二人生 (末日版)    14
神的孩子都在跳舞      13
为爱而生          13
自传            13
第一张创作专辑       12
爱情万岁          12
人生海海          12
后青春期的诗        12
Name: count, dtype: int64

In [8]:
# 纯音乐数量
instrumental_num = df_raw[df_raw['has_lyric'] ==
                          0]['album_name'].value_counts()
instrumental_num

album_name
为爱而生          2
神的孩子都在跳舞      1
第二人生 (明日版)    1
第二人生 (末日版)    1
知足 最真杰作选      1
Name: count, dtype: int64

In [9]:
# 新歌数量
new_songs_num = df_raw[df_raw['is_duplicate'] ==
                       0]['album_name'].value_counts()
new_songs_num

album_name
时光机           15
第二人生 (明日版)    14
神的孩子都在跳舞      13
为爱而生          13
自传            13
第一张创作专辑       12
爱情万岁          12
人生海海          12
后青春期的诗        12
知足 最真杰作选      12
步步 自选作品辑      10
第二人生 (末日版)     2
Name: count, dtype: int64

In [10]:
df_album = df_album.merge(songs_num, on='album_name', how='left').rename(columns={'count': 'songs_num'})
df_album

,index,album_id,album_name,album_type,release_date,album_fixed,songs_num
0,0,38315,第一张创作专辑,录音室专辑,1999-07-07,第一张创作专辑,12
1,12,38308,爱情万岁,录音室专辑,2000-07-07,爱情万岁,12
2,24,38297,人生海海,录音室专辑,2001-07-06,人生海海,12
3,36,38276,时光机,录音室专辑,2003-11-11,时光机,15
4,51,38259,神的孩子都在跳舞,录音室专辑,2004-11-05,神的孩子都在跳舞,13
5,64,38241,为爱而生,录音室专辑,2006-12-29,为爱而生,13
6,77,38235,后青春期的诗,录音室专辑,2008-10-23,后青春期的诗,12
7,89,2040001,第二人生 (明日版),录音室专辑,2011-12-16,第二人生,14
8,103,38214,第二人生 (末日版),录音室专辑,2011-12-16,第二人生,14
9,117,34746073,自传,录音室专辑,2016-07-21,自传,13


In [11]:
df_album = df_album.merge(
    instrumental_num, on='album_name',
    how='left').rename(columns={'count': 'instrumental_num'}).fillna(0)
df_album 

,index,album_id,album_name,album_type,release_date,album_fixed,songs_num,instrumental_num
0,0,38315,第一张创作专辑,录音室专辑,1999-07-07,第一张创作专辑,12,0.0
1,12,38308,爱情万岁,录音室专辑,2000-07-07,爱情万岁,12,0.0
2,24,38297,人生海海,录音室专辑,2001-07-06,人生海海,12,0.0
3,36,38276,时光机,录音室专辑,2003-11-11,时光机,15,0.0
4,51,38259,神的孩子都在跳舞,录音室专辑,2004-11-05,神的孩子都在跳舞,13,1.0
5,64,38241,为爱而生,录音室专辑,2006-12-29,为爱而生,13,2.0
6,77,38235,后青春期的诗,录音室专辑,2008-10-23,后青春期的诗,12,0.0
7,89,2040001,第二人生 (明日版),录音室专辑,2011-12-16,第二人生,14,1.0
8,103,38214,第二人生 (末日版),录音室专辑,2011-12-16,第二人生,14,1.0
9,117,34746073,自传,录音室专辑,2016-07-21,自传,13,0.0


In [12]:
df_album = df_album.merge(
    new_songs_num, on='album_name',
    how='left').rename(columns={'count': 'new_songs_num'}).fillna(0)
df_album 

,index,album_id,album_name,album_type,release_date,album_fixed,songs_num,instrumental_num,new_songs_num
0,0,38315,第一张创作专辑,录音室专辑,1999-07-07,第一张创作专辑,12,0.0,12
1,12,38308,爱情万岁,录音室专辑,2000-07-07,爱情万岁,12,0.0,12
2,24,38297,人生海海,录音室专辑,2001-07-06,人生海海,12,0.0,12
3,36,38276,时光机,录音室专辑,2003-11-11,时光机,15,0.0,15
4,51,38259,神的孩子都在跳舞,录音室专辑,2004-11-05,神的孩子都在跳舞,13,1.0,13
5,64,38241,为爱而生,录音室专辑,2006-12-29,为爱而生,13,2.0,13
6,77,38235,后青春期的诗,录音室专辑,2008-10-23,后青春期的诗,12,0.0,12
7,89,2040001,第二人生 (明日版),录音室专辑,2011-12-16,第二人生,14,1.0,14
8,103,38214,第二人生 (末日版),录音室专辑,2011-12-16,第二人生,14,1.0,2
9,117,34746073,自传,录音室专辑,2016-07-21,自传,13,0.0,13


In [13]:
album_dict = df_album.to_dict(orient='records')
with open('output/album_data.json', 'w', encoding='utf-8') as f:
    json.dump(album_dict, f, ensure_ascii=False, indent=4)


In [14]:
df_album['album_fixed'].tolist()

['第一张创作专辑',
 '爱情万岁',
 '人生海海',
 '时光机',
 '神的孩子都在跳舞',
 '为爱而生',
 '后青春期的诗',
 '第二人生',
 '第二人生',
 '自传',
 '知足 最真杰作选',
 '步步 自选作品辑']

## 曲目数据

In [15]:
df_songs = df_raw[df_raw['is_duplicate'] == 0][[
    'album_fixed', 'song_name'
]].drop_duplicates().reset_index(drop=True).copy()
df_songs

,album_fixed,song_name
0,第一张创作专辑,疯狂世界
1,第一张创作专辑,拥抱
2,第一张创作专辑,透露
3,第一张创作专辑,生活
4,第一张创作专辑,爱情的模样
...,...,...
135,步步 自选作品辑,离开地球表面
136,步步 自选作品辑,知足 (乐团版)
137,步步 自选作品辑,温柔 (2013Remix版)
138,步步 自选作品辑,你是唯一 (2013新录制作品)


In [16]:
songs_dict = df_songs.to_dict(orient='records')
with open('output/songs_data.json', 'w', encoding='utf-8') as f:
    json.dump(songs_dict, f, ensure_ascii=False, indent=4)

# 分词数据
名词： n, w
动词： v
形容词： v

In [17]:
df_lyric_raw

,Unnamed: 0,song_id,词,词性,频数,album_order,album_id,album_name,album_type,release_date,song_order,song_name,album_fixed,has_lyric,is_duplicate
0,0,386925,我,r,18,1,38315,第一张创作专辑,录音室专辑,1999-07-07,1,疯狂世界,第一张创作专辑,1,0
1,1,386925,好想,v,18,1,38315,第一张创作专辑,录音室专辑,1999-07-07,1,疯狂世界,第一张创作专辑,1,0
2,2,386925,那么,r,12,1,38315,第一张创作专辑,录音室专辑,1999-07-07,1,疯狂世界,第一张创作专辑,1,0
3,3,386925,多,m,12,1,38315,第一张创作专辑,录音室专辑,1999-07-07,1,疯狂世界,第一张创作专辑,1,0
4,4,386925,的,uj,12,1,38315,第一张创作专辑,录音室专辑,1999-07-07,1,疯狂世界,第一张创作专辑,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13380,13380,28181110,长大,ns,1,11,2740205,步步 自选作品辑,精选辑,2013-12-30,19,盛夏光年,步步 自选作品辑,1,0
13381,13381,28181110,难道,d,1,11,2740205,步步 自选作品辑,精选辑,2013-12-30,19,盛夏光年,步步 自选作品辑,1,0
13382,13382,28181110,人,n,1,11,2740205,步步 自选作品辑,精选辑,2013-12-30,19,盛夏光年,步步 自选作品辑,1,0
13383,13383,28181110,必经,d,1,11,2740205,步步 自选作品辑,精选辑,2013-12-30,19,盛夏光年,步步 自选作品辑,1,0


In [18]:
df_lyric_raw[df_lyric_raw['词'] == '用力']

,Unnamed: 0,song_id,词,词性,频数,album_order,album_id,album_name,album_type,release_date,song_order,song_name,album_fixed,has_lyric,is_duplicate
27,27,386925,用力,d,2,1,38315,第一张创作专辑,录音室专辑,1999-07-07,1,疯狂世界,第一张创作专辑,1,0
475,475,386932,用力,d,1,1,38315,第一张创作专辑,录音室专辑,1999-07-07,6,嘿！我要走了,第一张创作专辑,1,0
4927,4927,386183,用力,d,3,5,38259,神的孩子都在跳舞,录音室专辑,2004-11-05,6,约翰蓝侬,神的孩子都在跳舞,1,0
7299,7299,385798,用力,d,3,7,38235,后青春期的诗,录音室专辑,2008-10-23,8,春天的呐喊,后青春期的诗,1,0


In [19]:
df_lyric_raw['词'].value_counts()

词
的     132
我     127
是     109
你     104
了      91
     ... 
闹市      1
炊烟      1
混杂      1
小可      1
溃烂      1
Name: count, Length: 4823, dtype: int64

In [20]:
def word_count_by_pos(df, pos, words_num=30, within=True):
    if within:
        word_cnt = df[df['词性'].str.startswith(
            pos, na=False)]['词'].value_counts().reset_index()
        word_sum = df[df['词性'].str.startswith(
            pos, na=False)].groupby('词')['频数'].sum().reset_index()
    else:
        word_cnt = df[df['词性']==pos]['词'].value_counts().reset_index()
        word_sum = df[df['词性']==pos].groupby('词')['频数'].sum().reset_index()
    if words_num:
        res = word_cnt.head(words_num).merge(word_sum, on='词', how='left')
    else:
        res = word_cnt.merge(word_sum, on='词', how='left')
    res['order'] = 100 - res.index
    res = res.rename(columns={
        '词': 'word',
        'count': 'songs_num',
        '频数': 'frequency'
    })
    return res

In [21]:
pos_list = ['n', 'v', 'a']
words_dict = defaultdict(dict)
for pos in pos_list:
    res_df = word_count_by_pos(df_lyric_raw, pos, 70)
    words_dict[pos] = res_df.to_dict('list')

with open('output/words_data.json', 'w', encoding='utf-8') as f:
    json.dump(words_dict, f, ensure_ascii=False, indent=4)

# 词图

In [22]:
word_count_by_pos(df_lyric_raw, 'n', words_num=None, within=False)

,word,songs_num,frequency,order
0,世界,53,120,100
1,人,49,124,99
2,人生,32,81,98
3,爱情,29,51,97
4,梦,28,71,96
...,...,...,...,...
1346,敌,1,1,-1246
1347,超人,1,1,-1247
1348,灰烬,1,1,-1248
1349,超能力,1,1,-1249


In [24]:
word_count_by_pos(df_lyric_raw, 'n', words_num=None)

,word,songs_num,frequency,order
0,世界,53,120,100
1,人,49,124,99
2,人生,32,81,98
3,爱情,29,51,97
4,梦,28,71,96
...,...,...,...,...
1692,紫,1,1,-1592
1693,大雄,1,1,-1593
1694,越美越,1,1,-1594
1695,火花,1,1,-1595
